# Case-001 Advanced Therapy Referral Readiness Gate

This public-safe notebook models whether case-001 has enough records for specialist screening. It does not diagnose, select treatment, rank eligibility, or use private records.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

RecordStatus = Literal["missing", "partial", "ready"]


@dataclass(frozen=True)
class ReadinessArea:
    """One public-safe record area needed for specialist screening."""

    area: str
    status: RecordStatus
    screening_dependency: int
    safety_dependency: int
    access_dependency: int
    blocker: str

    @property
    def priority_score(self) -> int:
        """Rank missing areas by screening, safety, and access dependency."""
        if self.status == "ready":
            return 0
        weight = 3 if self.status == "missing" else 1
        return weight * (
            self.screening_dependency + self.safety_dependency + self.access_dependency
        )

In [ ]:
areas = [
    ReadinessArea(
        "current diagnosis, subtype, and genotype", "missing", 3, 2, 2, "phenotype_only"
    ),
    ReadinessArea(
        "annual transfusion burden and Hb response",
        "missing",
        3,
        2,
        1,
        "transfusion_dependent_burden_unquantified",
    ),
    ReadinessArea(
        "immune and blood-bank packet",
        "missing",
        3,
        3,
        1,
        "immune_transfusion_packet_missing",
    ),
    ReadinessArea(
        "iron, chelation, and organ-risk packet",
        "missing",
        3,
        3,
        1,
        "iron_packet_missing",
    ),
    ReadinessArea(
        "infection, vaccination, spleen, fertility, and counselling context",
        "missing",
        2,
        3,
        1,
        "organ_context_missing",
    ),
    ReadinessArea(
        "prior advanced-therapy review and access constraints",
        "missing",
        2,
        1,
        3,
        "access_history_missing",
    ),
]

ranked_areas = sorted(
    areas, key=lambda area: (area.priority_score, area.safety_dependency), reverse=True
)
[(area.area, area.priority_score, area.blocker) for area in ranked_areas]

In [ ]:
def classify_readiness(record_areas: list[ReadinessArea]) -> str:
    """Classify whether records can support specialist screening."""
    if any(area.status == "missing" for area in record_areas):
        return "advanced_therapy_referral_packet_missing"
    if any(area.status == "partial" for area in record_areas):
        return "advanced_therapy_referral_packet_partial"
    return "advanced_therapy_referral_packet_ready_for_specialist_screening"


result = {
    "case_id": "case-001",
    "current_label": classify_readiness(areas),
    "top_blockers": [area.blocker for area in ranked_areas[:3]],
    "patient_specific_claims": "blocked",
}
result

## Decision

Durable label: `advanced_therapy_referral_packet_missing`. HSCT, gene-cell therapy, disease-modifying drugs, and trials remain specialist-screening questions until the packet is complete enough for clinician review. Quranic motivation stays in the ethical-methods lane: expert consultation and measurement discipline, not biomedical proof.
